In [1]:
import math
import numpy as np

def _norm_cdf(x):
    """Standard normal CDF using math.erf (no scipy required)."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def black_scholes_price(S, K, T, r, sigma, option='call', q=0.0):
    """
    European Black-Scholes price.
    S: spot price
    K: strike
    T: time to maturity (in years)
    r: risk-free rate (annual, continuous)
    sigma: volatility (annual)
    option: 'call' or 'put'
    q: continuous dividend yield (default 0)
    """
    # - Funkcja liczy teoretyczną cenę europejskiej opcji (call lub put)
    #   według modelu Blacka‑Scholesa.
    # - S to bieżąca cena instrumentu (np. akcji), K to cena wykonania (strike).
    # - T to czas do wygaśnięcia w latach (np. 0.5 = 6 miesięcy).
    # - r to bezpieczna stopa procentowa (np. oprocentowanie obligacji),
    #   sigma to oczekiwana zmienność ceny (im wyższa, tym droższa opcja).
    # - q to ciągła stopa dywidendy (jeśli instrument wypłaca dywidendy).
    # - Wynik to ile teoretycznie warto zapłacić dzisiaj za daną opcję.
    # Przykład:
    #   black_scholes_price(100, 100, 1.0, 0.01, 0.2, 'call')
    if T <= 0 or sigma <= 0:
        # immediate payoff if expired or zero vol
        if option.lower().startswith('c'):
            return max(S - K, 0.0)
        return max(K - S, 0.0)

    d1 = (math.log(S / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    if option.lower().startswith('c'):
        return S * math.exp(-q * T) * _norm_cdf(d1) - K * math.exp(-r * T) * _norm_cdf(d2)
    return K * math.exp(-r * T) * _norm_cdf(-d2) - S * math.exp(-q * T) * _norm_cdf(-d1)

# Example:
price_call = black_scholes_price(S=100, K=100, T=1.0, r=0.01, sigma=0.2, option='call')
price_call
price_put  = black_scholes_price(S=100, K=100, T=1.0, r=0.01, sigma=0.2, option='put')

In [5]:
S = 100.0
K = 110.0
T = 5.0
r = 0.01
sigma = 0.2
q = 0.0

# Ustawienia Monte Carlo
np.random.seed(42)
N = 200_000  # liczba symulacji

# Symulacja cen końcowych wg GBM
Z = np.random.normal(size=N)
ST = S * np.exp((r - q - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

# Payoffy i dyskontowanie
call_payoffs = np.maximum(ST - K, 0.0)
put_payoffs = np.maximum(K - ST, 0.0)
discount = np.exp(-r * T)

call_price_mc = discount * call_payoffs.mean()
put_price_mc = discount * put_payoffs.mean()

# Błędy standardowe i przedziały ufności 95%
call_se = discount * call_payoffs.std(ddof=1) / np.sqrt(N)
put_se = discount * put_payoffs.std(ddof=1) / np.sqrt(N)
z = 1.96
call_ci = (call_price_mc - z * call_se, call_price_mc + z * call_se)
put_ci = (put_price_mc - z * put_se, put_price_mc + z * put_se)

# Wyniki porównane z analitycznymi price_call / price_put
print(f"MC call: {call_price_mc:.6f} ± {z*call_se:.6f} (95% CI {call_ci[0]:.6f}, {call_ci[1]:.6f})")
print(f"Analityczny call: {price_call:.6f}  Różnica: {call_price_mc - price_call:.6f}\n")
print(f"MC put:  {put_price_mc:.6f} ± {z*put_se:.6f} (95% CI {put_ci[0]:.6f}, {put_ci[1]:.6f})")
print(f"Analityczny put:  {price_put:.6f}  Różnica: {put_price_mc - price_put:.6f}")

MC call: 15.890844 ± 0.143815 (95% CI 15.747029, 16.034659)
Analityczny call: 8.433319  Różnica: 7.457526

MC put:  20.492317 ± 0.095807 (95% CI 20.396511, 20.588124)
Analityczny put:  7.438302  Różnica: 13.054015


# Wyjaśnienie kodu (komórki z funkcją i symulacją Monte Carlo)

Poniżej znajduje się szczegółowe objaśnienie krok po kroku.

## Funkcje i stałe (CELL INDEX: 0)
- _norm_cdf(x)
    - Zwraca dystrybuantę standardowego rozkładu normalnego za pomocą funkcji error function: Φ(x) = 0.5 * (1 + erf(x / √2)).
    - Dzięki temu nie potrzebujemy scipy.

- black_scholes_price(S, K, T, r, sigma, option='call', q=0.0)
    - Oblicza teoretyczną cenę europejskiej opcji według modelu Black‑Scholes (call lub put).
    - Parametry:
        - S: cena spot
        - K: strike
        - T: czas do wygaśnięcia (w latach)
        - r: stopa wolna od ryzyka (ciągłe kapitalizowanie)
        - sigma: zmienność roczna
        - q: ciągły yield/dywidenda
    - Obsługa brzegowa:
        - Jeśli T <= 0 lub sigma <= 0, funkcja zwraca natychmiastowy payoff (max(S-K,0) dla call, max(K-S,0) dla put).
    - Wzory:
        - d1 = (ln(S/K) + (r - q + 0.5*sigma^2) * T) / (sigma * sqrt(T))
        - d2 = d1 - sigma * sqrt(T)
        - Call: S e^{-qT} Φ(d1) − K e^{-rT} Φ(d2)
        - Put: K e^{-rT} Φ(−d2) − S e^{-qT} Φ(−d1)

- Przykład: price_call i price_put obliczone na końcu komórki (użyte później jako odniesienie).

## Symulacja Monte Carlo (CELL INDEX: 1)
Cel: oszacować cenę europejskich opcji przez symulację końcowych cen akcji zgodnie z Geometric Brownian Motion (GBM) i porównać z cenami analitycznymi.

1. Parametry wejściowe
     - S, K, T, r, sigma, q ustawione na typowe wartości (tu S=K=100, T=1, r=0.01, sigma=0.2, q=0).
     - N = 200_000 — liczba ścieżek symulacji.
     - np.random.seed(42) — ustawia ziarno losowości, zapewnia powtarzalność wyników.

2. Symulacja cen końcowych ST (wektorowa operacja numpy)
     - Z = np.random.normal(size=N) — N próbek z N(0,1).
     - Model GBM (wzór analityczny dla ceny w czasie T):
         ST = S * exp((r - q - 0.5 * sigma^2) * T + sigma * sqrt(T) * Z)
     - To jest bezpośrednia symulacja log‑normalnego rozkładu wynikającego z modelu Blacka‑Scholesa.

3. Obliczanie payoffów
     - call_payoffs = np.maximum(ST - K, 0.0)
     - put_payoffs  = np.maximum(K - ST, 0.0)
     - Payoffy są wektorami o długości N.

4. Dyskontowanie do wartości bieżącej
     - discount = exp(-r * T)
     - Szacowane ceny:
         - call_price_mc = discount * mean(call_payoffs)
         - put_price_mc  = discount * mean(put_payoffs)

5. Estymacja niepewności (błąd standardowy i 95% CI)
     - Standard error (dla ceny z dyskontowaniem): se = discount * std(sample, ddof=1) / sqrt(N)
         - ddof=1 używa niewyrownanej estymacji odchylenia standardowego (sample std).
     - CI 95%: (estimate − z*se, estimate + z*se) z z = 1.96.

6. Porównanie z wartością analityczną
     - Drukowany wynik porównuje call_price_mc z price_call (analityczny obliczony przez black_scholes_price).
     - W przykładzie uzyskane wartości to:
         - call_price_mc ≈ 8.442616622574866
         - price_call (analityczny) = 8.433318690109608
         - 95% CI dla call: (≈ 8.38360411, 8.50162914)
         - Różnica ≈ 0.0093 (czyli wynik symulacji jest bliski wartości analitycznej, zmienność zgodna z CI).

## Uwagi praktyczne
- Symulacja jest wektorowa (numpy) → szybka przy dużych N.
- Ustawienie ziarna (seed) daje powtarzalność wyników.
- N=200k daje stosunkowo mały błąd standardowy; można zmniejszyć błąd zwiększając N lub stosując techniki redukcji wariancji (control variates, antithetic variates).
- Dla T <= 0 lub sigma <= 0 funkcja zwraca bezpośredni payoff — ważne zabezpieczenie przed dzieleniem przez zero.
- Porównanie MC z analitycznym jest użytecznym testem poprawności implementacji.

Jeśli chcesz, mogę:
- dodać wariant z redukcją wariancji (np. antithetic),
- policzyć grecki (delta, vega) numerycznie z tej samej symulacji,
- narysować histogram ST lub rozkład payoffów.

In [6]:
print(f"MC call: {call_price_mc:.6f} ± {z*call_se:.6f} (95% CI {call_ci[0]:.6f}, {call_ci[1]:.6f})")
print(f"Analityczny call: {price_call:.6f}  Różnica: {call_price_mc - price_call:.6f}\n")
print(f"MC put:  {put_price_mc:.6f} ± {z*put_se:.6f} (95% CI {put_ci[0]:.6f}, {put_ci[1]:.6f})")
print(f"Analityczny put:  {price_put:.6f}  Różnica: {put_price_mc - price_put:.6f}")

MC call: 15.890844 ± 0.143815 (95% CI 15.747029, 16.034659)
Analityczny call: 8.433319  Różnica: 7.457526

MC put:  20.492317 ± 0.095807 (95% CI 20.396511, 20.588124)
Analityczny put:  7.438302  Różnica: 13.054015


In [7]:
import math
from scipy.stats import norm


def black_scholes_call_put(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float
) -> tuple[float, float]:
    """
    Zwraca cenę europejskiej opcji call i put w modelu Blacka-Scholesa.

    Parametry:
    S0    - aktualna cena instrumentu bazowego
    K     - cena wykonania
    T     - czas do wygaśnięcia w latach
    r     - stopa wolna od ryzyka
    sigma - zmienność

    Zwraca:
    (call_price, put_price)
    """

    if S0 <= 0:
        raise ValueError("S0 musi być dodatnie.")
    if K <= 0:
        raise ValueError("K musi być dodatnie.")
    if T < 0:
        raise ValueError("T nie może być ujemne.")
    if sigma < 0:
        raise ValueError("sigma nie może być ujemne.")

    if T == 0:
        call = max(S0 - K, 0.0)
        put = max(K - S0, 0.0)
        return call, put

    if sigma == 0:
        discounted_strike = K * math.exp(-r * T)
        call = max(S0 - discounted_strike, 0.0)
        put = max(discounted_strike - S0, 0.0)
        return call, put

    sqrt_T = math.sqrt(T)

    d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T

    call = S0 * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    put = K * math.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)

    return call, put


# Przykład użycia
S0 = 100
K = 100
T = 1
r = 0.05
sigma = 0.2

call_price, put_price = black_scholes_call_put(S0, K, T, r, sigma)

print(f"Call price: {call_price:.4f}")
print(f"Put price:  {put_price:.4f}")

Call price: 10.4506
Put price:  5.5735


In [9]:
import math
import numpy as np
from scipy.stats import norm


def bs_call(S0: float, K: float, T: float, r: float, sigma: float) -> float:
    d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S0 * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)


def bs_put(S0: float, K: float, T: float, r: float, sigma: float) -> float:
    d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return K * math.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)


def mc_european_call(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    n_sim: int = 100_000,
    seed: int | None = 42
) -> dict:
    """
    Monte Carlo dla europejskiej opcji call w modelu Blacka-Scholesa.
    Zwraca estymatę ceny, odchylenie standardowe estymatora i 95% CI.
    """

    rng = np.random.default_rng(seed)

    Z = rng.standard_normal(n_sim)

    # Dokładny rozkład S_T w modelu BS:
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z)

    payoffs = np.maximum(ST - K, 0.0)
    discounted_payoffs = np.exp(-r * T) * payoffs

    price_mc = discounted_payoffs.mean()
    sample_std = discounted_payoffs.std(ddof=1)
    std_error = sample_std / math.sqrt(n_sim)

    ci_low = price_mc - 1.96 * std_error
    ci_high = price_mc + 1.96 * std_error

    return {
        "price_mc": price_mc,
        "sample_std": sample_std,
        "std_error": std_error,
        "ci_95": (ci_low, ci_high),
    }


def mc_european_put(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    n_sim: int = 100_000,
    seed: int | None = 42
) -> dict:
    """
    Monte Carlo dla europejskiej opcji put w modelu Blacka-Scholesa.
    Zwraca estymatę ceny, odchylenie standardowe estymatora i 95% CI.
    """

    rng = np.random.default_rng(seed)

    Z = rng.standard_normal(n_sim)

    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z)

    payoffs = np.maximum(K - ST, 0.0)
    discounted_payoffs = np.exp(-r * T) * payoffs

    price_mc = discounted_payoffs.mean()
    sample_std = discounted_payoffs.std(ddof=1)
    std_error = sample_std / math.sqrt(n_sim)

    ci_low = price_mc - 1.96 * std_error
    ci_high = price_mc + 1.96 * std_error

    return {
        "price_mc": price_mc,
        "sample_std": sample_std,
        "std_error": std_error,
        "ci_95": (ci_low, ci_high),
    }


if __name__ == "__main__":
    S0 = 100.0
    K = 110.0
    T = 5.0
    r = 0.05
    sigma = 0.2
    n_sim = 200_000

    call_bs = bs_call(S0, K, T, r, sigma)
    put_bs = bs_put(S0, K, T, r, sigma)

    call_mc = mc_european_call(S0, K, T, r, sigma, n_sim=n_sim, seed=123)
    put_mc = mc_european_put(S0, K, T, r, sigma, n_sim=n_sim, seed=123)

    print("=== CALL ===")
    print(f"Black-Scholes: {call_bs:.6f}")
    print(f"Monte Carlo:   {call_mc['price_mc']:.6f}")
    print(f"Std error:     {call_mc['std_error']:.6f}")
    print(f"95% CI:        ({call_mc['ci_95'][0]:.6f}, {call_mc['ci_95'][1]:.6f})")
    print(f"Abs error:     {abs(call_bs - call_mc['price_mc']):.6f}")

    print("\n=== PUT ===")
    print(f"Black-Scholes: {put_bs:.6f}")
    print(f"Monte Carlo:   {put_mc['price_mc']:.6f}")
    print(f"Std error:     {put_mc['std_error']:.6f}")
    print(f"95% CI:        ({put_mc['ci_95'][0]:.6f}, {put_mc['ci_95'][1]:.6f})")
    print(f"Abs error:     {abs(put_bs - put_mc['price_mc']):.6f}")

=== CALL ===
Black-Scholes: 24.546170
Monte Carlo:   24.490914
Std error:     0.086252
95% CI:        (24.321859, 24.659968)
Abs error:     0.055256

=== PUT ===
Black-Scholes: 10.214256
Monte Carlo:   10.219278
Std error:     0.033336
95% CI:        (10.153939, 10.284617)
Abs error:     0.005022


In [10]:
import math
import numpy as np
from scipy.stats import norm


def black_scholes_call_put(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float
) -> tuple[float, float]:
    """
    Ceny europejskiej opcji call i put w modelu Blacka-Scholesa.
    """

    if S0 <= 0:
        raise ValueError("S0 musi być dodatnie.")
    if K <= 0:
        raise ValueError("K musi być dodatnie.")
    if T < 0:
        raise ValueError("T nie może być ujemne.")
    if sigma < 0:
        raise ValueError("sigma nie może być ujemne.")

    if T == 0:
        call = max(S0 - K, 0.0)
        put = max(K - S0, 0.0)
        return call, put

    if sigma == 0:
        discounted_strike = K * math.exp(-r * T)
        call = max(S0 - discounted_strike, 0.0)
        put = max(discounted_strike - S0, 0.0)
        return call, put

    sqrt_T = math.sqrt(T)

    d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T

    call = S0 * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    put = K * math.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)

    return call, put


def monte_carlo_european_options(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    n_sim: int = 100_000,
    seed: int | None = 42
) -> dict:
    """
    Wycena europejskiej opcji call i put metodą Monte Carlo
    w modelu Blacka-Scholesa.

    Zakładamy ruch geometryczny Browna pod miarą neutralną względem ryzyka:
    S_T = S_0 * exp((r - 0.5 sigma^2)T + sigma sqrt(T) Z),
    gdzie Z ~ N(0,1).
    """

    if S0 <= 0:
        raise ValueError("S0 musi być dodatnie.")
    if K <= 0:
        raise ValueError("K musi być dodatnie.")
    if T < 0:
        raise ValueError("T nie może być ujemne.")
    if sigma < 0:
        raise ValueError("sigma nie może być ujemne.")
    if n_sim <= 1:
        raise ValueError("n_sim musi być większe niż 1.")

    if seed is not None:
        np.random.seed(seed)

    if T == 0:
        call = max(S0 - K, 0.0)
        put = max(K - S0, 0.0)
        return {
            "call_price_mc": call,
            "put_price_mc": put,
            "call_std_error": 0.0,
            "put_std_error": 0.0,
            "call_ci_95": (call, call),
            "put_ci_95": (put, put),
        }

    Z = np.random.normal(0.0, 1.0, n_sim)

    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z)

    discount_factor = math.exp(-r * T)

    call_payoffs = np.maximum(ST - K, 0.0)
    put_payoffs = np.maximum(K - ST, 0.0)

    discounted_call_payoffs = discount_factor * call_payoffs
    discounted_put_payoffs = discount_factor * put_payoffs

    call_price_mc = np.mean(discounted_call_payoffs)
    put_price_mc = np.mean(discounted_put_payoffs)

    call_std_error = np.std(discounted_call_payoffs, ddof=1) / math.sqrt(n_sim)
    put_std_error = np.std(discounted_put_payoffs, ddof=1) / math.sqrt(n_sim)

    z_975 = norm.ppf(0.975)

    call_ci_95 = (
        call_price_mc - z_975 * call_std_error,
        call_price_mc + z_975 * call_std_error
    )
    put_ci_95 = (
        put_price_mc - z_975 * put_std_error,
        put_price_mc + z_975 * put_std_error
    )

    return {
        "call_price_mc": call_price_mc,
        "put_price_mc": put_price_mc,
        "call_std_error": call_std_error,
        "put_std_error": put_std_error,
        "call_ci_95": call_ci_95,
        "put_ci_95": put_ci_95,
    }


def compare_bs_vs_mc(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    n_sim: int = 100_000,
    seed: int | None = 42
) -> dict:
    """
    Porównanie wyniku dokładnego Black-Scholes z wynikiem Monte Carlo.
    """

    call_bs, put_bs = black_scholes_call_put(S0, K, T, r, sigma)
    mc_result = monte_carlo_european_options(S0, K, T, r, sigma, n_sim=n_sim, seed=seed)

    return {
        "call_bs": call_bs,
        "put_bs": put_bs,
        "call_mc": mc_result["call_price_mc"],
        "put_mc": mc_result["put_price_mc"],
        "call_abs_error": abs(call_bs - mc_result["call_price_mc"]),
        "put_abs_error": abs(put_bs - mc_result["put_price_mc"]),
        "call_std_error": mc_result["call_std_error"],
        "put_std_error": mc_result["put_std_error"],
        "call_ci_95": mc_result["call_ci_95"],
        "put_ci_95": mc_result["put_ci_95"],
    }


# Przykład użycia
if __name__ == "__main__":
    S0 = 100
    K = 100
    T = 1
    r = 0.05
    sigma = 0.2
    n_sim = 100_000

    result = compare_bs_vs_mc(S0, K, T, r, sigma, n_sim=n_sim, seed=42)

    print("=== Black-Scholes vs Monte Carlo ===")
    print(f"Call BS:           {result['call_bs']:.6f}")
    print(f"Call MC:           {result['call_mc']:.6f}")
    print(f"Call abs error:    {result['call_abs_error']:.6f}")
    print(f"Call std error:    {result['call_std_error']:.6f}")
    print(f"Call 95% CI:       ({result['call_ci_95'][0]:.6f}, {result['call_ci_95'][1]:.6f})")
    print()
    print(f"Put BS:            {result['put_bs']:.6f}")
    print(f"Put MC:            {result['put_mc']:.6f}")
    print(f"Put abs error:     {result['put_abs_error']:.6f}")
    print(f"Put std error:     {result['put_std_error']:.6f}")
    print(f"Put 95% CI:        ({result['put_ci_95'][0]:.6f}, {result['put_ci_95'][1]:.6f})")

=== Black-Scholes vs Monte Carlo ===
Call BS:           10.450584
Call MC:           10.473892
Call abs error:    0.023308
Call std error:    0.046589
Call 95% CI:       (10.382579, 10.565205)

Put BS:            5.573526
Put MC:            5.574186
Put abs error:     0.000660
Put std error:     0.027382
Put 95% CI:        (5.520519, 5.627852)


In [12]:
simulation_sizes = [1_000, 5_000, 10_000, 50_000, 100_000]

S0 = 100
K = 100
T = 1
r = 0.03
sigma = 0.6

call_bs, put_bs = black_scholes_call_put(S0, K, T, r, sigma)

print(f"Call BS exact = {call_bs:.6f}")
print(f"Put  BS exact = {put_bs:.6f}")
print()

for n_sim in simulation_sizes:
    result = compare_bs_vs_mc(S0, K, T, r, sigma, n_sim=n_sim, seed=42)
    print(f"n_sim = {n_sim:>6}")
    print(f"  Call MC       = {result['call_mc']:.6f}")
    print(f"  Call error    = {result['call_abs_error']:.6f}")
    print(f"  Put MC        = {result['put_mc']:.6f}")
    print(f"  Put error     = {result['put_abs_error']:.6f}")
    print()

Call BS exact = 24.739700
Put  BS exact = 21.784253

n_sim =   1000
  Call MC       = 25.023883
  Call error    = 0.284183
  Put MC        = 21.228601
  Put error     = 0.555652

n_sim =   5000
  Call MC       = 24.673395
  Call error    = 0.066305
  Put MC        = 21.520642
  Put error     = 0.263611

n_sim =  10000
  Call MC       = 24.812378
  Call error    = 0.072678
  Put MC        = 21.841673
  Put error     = 0.057420

n_sim =  50000
  Call MC       = 24.693782
  Call error    = 0.045918
  Put MC        = 21.794172
  Put error     = 0.009919

n_sim = 100000
  Call MC       = 24.810308
  Call error    = 0.070608
  Put MC        = 21.776841
  Put error     = 0.007413



In [14]:
import math
import numpy as np
from scipy.stats import norm


# ============================================================
# 1. WZÓR BLACKA-SCHOLESA
# ============================================================

def black_scholes_call_put(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float
) -> tuple[float, float]:
    """
    Liczy ceny europejskiej opcji call i put w modelu Blacka-Scholesa.

    Parametry:
    ----------
    S0 : float
        Aktualna cena instrumentu bazowego w chwili t=0.
    K : float
        Cena wykonania opcji (strike).
    T : float
        Czas do wygaśnięcia opcji, wyrażony w latach.
    r : float
        Ciągła stopa wolna od ryzyka.
    sigma : float
        Zmienność instrumentu bazowego.

    Zwraca:
    -------
    (call_price, put_price) : tuple[float, float]
        Cena opcji call oraz put.
    """

    # Podstawowa walidacja parametrów wejściowych
    if S0 <= 0:
        raise ValueError("S0 musi być dodatnie.")
    if K <= 0:
        raise ValueError("K musi być dodatnie.")
    if T < 0:
        raise ValueError("T nie może być ujemne.")
    if sigma < 0:
        raise ValueError("sigma nie może być ujemne.")

    # Jeśli czas do wygaśnięcia wynosi 0,
    # to opcja ma wartość równą natychmiastowemu payoffowi.
    if T == 0:
        call = max(S0 - K, 0.0)
        put = max(K - S0, 0.0)
        return call, put

    # Jeśli zmienność sigma = 0,
    # to proces nie jest losowy i można policzyć wartość deterministycznie.
    if sigma == 0:
        discounted_strike = K * math.exp(-r * T)
        call = max(S0 - discounted_strike, 0.0)
        put = max(discounted_strike - S0, 0.0)
        return call, put

    # Pomocnicze wielkości
    sqrt_T = math.sqrt(T)

    # Standardowe definicje d1 i d2 w modelu Blacka-Scholesa
    d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T

    # Cena opcji call
    call = S0 * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)

    # Cena opcji put
    put = K * math.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)

    return call, put


# ============================================================
# 2. MONTE CARLO BEZ dt
# ============================================================

def monte_carlo_european_options_direct(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    n_sim: int = 100_000,
    seed: int | None = 42
) -> dict:
    """
    Wycena europejskiej opcji call i put metodą Monte Carlo
    przez bezpośrednie losowanie końcowej wartości S_T.

    To podejście NIE używa dt ani symulacji ścieżki.
    Jest bardzo dobre dla zwykłej opcji europejskiej, bo w modelu
    Blacka-Scholesa znamy dokładny rozkład S_T.

    Model pod miarą neutralną względem ryzyka:
        S_T = S0 * exp((r - 0.5*sigma^2)*T + sigma*sqrt(T)*Z),
    gdzie Z ~ N(0,1).

    Zwraca słownik z cenami, błędami standardowymi i przedziałami ufności.
    """

    # Walidacja
    if S0 <= 0:
        raise ValueError("S0 musi być dodatnie.")
    if K <= 0:
        raise ValueError("K musi być dodatnie.")
    if T < 0:
        raise ValueError("T nie może być ujemne.")
    if sigma < 0:
        raise ValueError("sigma nie może być ujemne.")
    if n_sim <= 1:
        raise ValueError("n_sim musi być większe niż 1.")

    # Ustawienie ziarna generatora liczb losowych
    if seed is not None:
        np.random.seed(seed)

    # Przypadek T = 0
    if T == 0:
        call = max(S0 - K, 0.0)
        put = max(K - S0, 0.0)
        return {
            "call_price_mc": call,
            "put_price_mc": put,
            "call_std_error": 0.0,
            "put_std_error": 0.0,
            "call_ci_95": (call, call),
            "put_ci_95": (put, put),
        }

    # Losujemy n_sim zmiennych standardowo normalnych
    Z = np.random.normal(loc=0.0, scale=1.0, size=n_sim)

    # Bezpośrednie losowanie końcowej ceny S_T
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z)

    # Payoffy dla call i put
    call_payoffs = np.maximum(ST - K, 0.0)
    put_payoffs = np.maximum(K - ST, 0.0)

    # Dyskontowanie payoffów do chwili 0
    discount_factor = math.exp(-r * T)
    discounted_call_payoffs = discount_factor * call_payoffs
    discounted_put_payoffs = discount_factor * put_payoffs

    # Estymatory cen opcji
    call_price_mc = np.mean(discounted_call_payoffs)
    put_price_mc = np.mean(discounted_put_payoffs)

    # Błąd standardowy estymatora średniej
    call_std_error = np.std(discounted_call_payoffs, ddof=1) / math.sqrt(n_sim)
    put_std_error = np.std(discounted_put_payoffs, ddof=1) / math.sqrt(n_sim)

    # 95% przedział ufności przy użyciu kwantyla N(0,1)
    z_975 = norm.ppf(0.975)

    call_ci_95 = (
        call_price_mc - z_975 * call_std_error,
        call_price_mc + z_975 * call_std_error
    )
    put_ci_95 = (
        put_price_mc - z_975 * put_std_error,
        put_price_mc + z_975 * put_std_error
    )

    return {
        "call_price_mc": call_price_mc,
        "put_price_mc": put_price_mc,
        "call_std_error": call_std_error,
        "put_std_error": put_std_error,
        "call_ci_95": call_ci_95,
        "put_ci_95": put_ci_95,
    }


# ============================================================
# 3. MONTE CARLO Z dt
# ============================================================

def monte_carlo_european_options_with_dt(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    n_sim: int = 100_000,
    n_steps: int = 100,
    seed: int | None = 42
) -> dict:
    """
    Wycena europejskiej opcji call i put metodą Monte Carlo
    z dyskretyzacją czasu, czyli z krokiem dt = T / n_steps.

    Tutaj symulujemy pełną ścieżkę:
        S_{t+dt} = S_t * exp((r - 0.5*sigma^2)dt + sigma*sqrt(dt)*Z)

    To podejście ma sens, gdy:
    - chcesz oglądać trajektorie,
    - payoff zależy od całej ścieżki,
    - chcesz jawnie mieć dt w modelu.

    Dla zwykłej opcji europejskiej jest zwykle wolniejsze niż losowanie
    bezpośredniego S_T, ale bywa potrzebne dydaktycznie.
    """

    # Walidacja
    if S0 <= 0:
        raise ValueError("S0 musi być dodatnie.")
    if K <= 0:
        raise ValueError("K musi być dodatnie.")
    if T < 0:
        raise ValueError("T nie może być ujemne.")
    if sigma < 0:
        raise ValueError("sigma nie może być ujemne.")
    if n_sim <= 1:
        raise ValueError("n_sim musi być większe niż 1.")
    if n_steps <= 0:
        raise ValueError("n_steps musi być dodatnie.")

    # Ustawienie ziarna losowości
    if seed is not None:
        np.random.seed(seed)

    # Jeśli T = 0, zwracamy payoff natychmiastowy
    if T == 0:
        call = max(S0 - K, 0.0)
        put = max(K - S0, 0.0)
        return {
            "call_price_mc": call,
            "put_price_mc": put,
            "call_std_error": 0.0,
            "put_std_error": 0.0,
            "call_ci_95": (call, call),
            "put_ci_95": (put, put),
            "dt": 0.0,
        }

    # Krok czasowy
    dt = T / n_steps
    sqrt_dt = math.sqrt(dt)

    # Każda ścieżka startuje z S0
    S = np.full(shape=n_sim, fill_value=S0, dtype=float)

    # Iteracyjna symulacja ścieżki
    for _ in range(n_steps):
        # Nowy zestaw zmiennych losowych dla każdego kroku
        Z = np.random.normal(loc=0.0, scale=1.0, size=n_sim)

        # Aktualizacja ceny aktywa zgodnie z log-Eulerem
        S *= np.exp((r - 0.5 * sigma**2) * dt + sigma * sqrt_dt * Z)

    # Po zakończeniu pętli S zawiera końcowe wartości S_T
    ST = S

    # Payoffy opcji
    call_payoffs = np.maximum(ST - K, 0.0)
    put_payoffs = np.maximum(K - ST, 0.0)

    # Dyskontowanie
    discount_factor = math.exp(-r * T)
    discounted_call_payoffs = discount_factor * call_payoffs
    discounted_put_payoffs = discount_factor * put_payoffs

    # Estymatory cen
    call_price_mc = np.mean(discounted_call_payoffs)
    put_price_mc = np.mean(discounted_put_payoffs)

    # Błędy standardowe
    call_std_error = np.std(discounted_call_payoffs, ddof=1) / math.sqrt(n_sim)
    put_std_error = np.std(discounted_put_payoffs, ddof=1) / math.sqrt(n_sim)

    # 95% przedziały ufności
    z_975 = norm.ppf(0.975)

    call_ci_95 = (
        call_price_mc - z_975 * call_std_error,
        call_price_mc + z_975 * call_std_error
    )
    put_ci_95 = (
        put_price_mc - z_975 * put_std_error,
        put_price_mc + z_975 * put_std_error
    )

    return {
        "call_price_mc": call_price_mc,
        "put_price_mc": put_price_mc,
        "call_std_error": call_std_error,
        "put_std_error": put_std_error,
        "call_ci_95": call_ci_95,
        "put_ci_95": put_ci_95,
        "dt": dt,
    }


# ============================================================
# 4. FUNKCJA PORÓWNUJĄCA WSZYSTKO
# ============================================================

def compare_methods(
    S0: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    n_sim: int = 100_000,
    n_steps: int = 100,
    seed: int | None = 42
) -> dict:
    """
    Porównuje:
    - dokładny wzór Blacka-Scholesa,
    - Monte Carlo bez dt,
    - Monte Carlo z dt.

    Zwraca wszystkie najważniejsze wyniki w jednym słowniku.
    """

    # Wynik dokładny
    call_bs, put_bs = black_scholes_call_put(S0, K, T, r, sigma)

    # Monte Carlo bez symulacji ścieżki
    mc_direct = monte_carlo_european_options_direct(
        S0=S0,
        K=K,
        T=T,
        r=r,
        sigma=sigma,
        n_sim=n_sim,
        seed=seed
    )

    # Monte Carlo z krokiem dt
    mc_dt = monte_carlo_european_options_with_dt(
        S0=S0,
        K=K,
        T=T,
        r=r,
        sigma=sigma,
        n_sim=n_sim,
        n_steps=n_steps,
        seed=seed
    )

    return {
        "call_bs": call_bs,
        "put_bs": put_bs,

        "call_mc_direct": mc_direct["call_price_mc"],
        "put_mc_direct": mc_direct["put_price_mc"],
        "call_mc_direct_error": abs(call_bs - mc_direct["call_price_mc"]),
        "put_mc_direct_error": abs(put_bs - mc_direct["put_price_mc"]),
        "call_mc_direct_ci_95": mc_direct["call_ci_95"],
        "put_mc_direct_ci_95": mc_direct["put_ci_95"],

        "call_mc_dt": mc_dt["call_price_mc"],
        "put_mc_dt": mc_dt["put_price_mc"],
        "call_mc_dt_error": abs(call_bs - mc_dt["call_price_mc"]),
        "put_mc_dt_error": abs(put_bs - mc_dt["put_price_mc"]),
        "call_mc_dt_ci_95": mc_dt["call_ci_95"],
        "put_mc_dt_ci_95": mc_dt["put_ci_95"],
        "dt": mc_dt["dt"],
    }


# ============================================================
# 5. PRZYKŁAD URUCHOMIENIA
# ============================================================

if __name__ == "__main__":
    # Parametry przykładowe
    S0 = 100.0      # cena początkowa instrumentu
    K = 110.0       # strike
    T = 5.0         # 1 rok do wygaśnięcia
    r = 0.05        # 5% stopa wolna od ryzyka
    sigma = 0.6     # 20% zmienność

    n_sim = 100_000
    n_steps = 100

    # Porównanie metod
    result = compare_methods(
        S0=S0,
        K=K,
        T=T,
        r=r,
        sigma=sigma,
        n_sim=n_sim,
        n_steps=n_steps,
        seed=42
    )

    print("===================================================")
    print("PORÓWNANIE: BLACK-SCHOLES vs MONTE CARLO")
    print("===================================================")
    print(f"S0      = {S0}")
    print(f"K       = {K}")
    print(f"T       = {T}")
    print(f"r       = {r}")
    print(f"sigma   = {sigma}")
    print(f"n_sim   = {n_sim}")
    print(f"n_steps = {n_steps}")
    print(f"dt      = {result['dt']}")
    print()

    print("----- Wzór Blacka-Scholesa -----")
    print(f"Call BS = {result['call_bs']:.6f}")
    print(f"Put  BS = {result['put_bs']:.6f}")
    print()

    print("----- Monte Carlo bez dt (bezpośrednie losowanie S_T) -----")
    print(f"Call MC direct = {result['call_mc_direct']:.6f}")
    print(f"Put  MC direct = {result['put_mc_direct']:.6f}")
    print(f"Call abs error = {result['call_mc_direct_error']:.6f}")
    print(f"Put  abs error = {result['put_mc_direct_error']:.6f}")
    print(
        f"Call 95% CI    = "
        f"({result['call_mc_direct_ci_95'][0]:.6f}, {result['call_mc_direct_ci_95'][1]:.6f})"
    )
    print(
        f"Put  95% CI    = "
        f"({result['put_mc_direct_ci_95'][0]:.6f}, {result['put_mc_direct_ci_95'][1]:.6f})"
    )
    print()

    print("----- Monte Carlo z dt (symulacja ścieżki) -----")
    print(f"Call MC dt = {result['call_mc_dt']:.6f}")
    print(f"Put  MC dt = {result['put_mc_dt']:.6f}")
    print(f"Call abs error = {result['call_mc_dt_error']:.6f}")
    print(f"Put  abs error = {result['put_mc_dt_error']:.6f}")
    print(
        f"Call 95% CI    = "
        f"({result['call_mc_dt_ci_95'][0]:.6f}, {result['call_mc_dt_ci_95'][1]:.6f})"
    )
    print(
        f"Put  95% CI    = "
        f"({result['put_mc_dt_ci_95'][0]:.6f}, {result['put_mc_dt_ci_95'][1]:.6f})"
    )

PORÓWNANIE: BLACK-SCHOLES vs MONTE CARLO
S0      = 100.0
K       = 110.0
T       = 5.0
r       = 0.05
sigma   = 0.6
n_sim   = 100000
n_steps = 100
dt      = 0.05

----- Wzór Blacka-Scholesa -----
Call BS = 53.628998
Put  BS = 39.297084

----- Monte Carlo bez dt (bezpośrednie losowanie S_T) -----
Call MC direct = 53.639250
Put  MC direct = 39.288028
Call abs error = 0.010253
Put  abs error = 0.009056
Call 95% CI    = (52.358523, 54.919978)
Put  95% CI    = (39.093845, 39.482212)

----- Monte Carlo z dt (symulacja ścieżki) -----
Call MC dt = 52.730434
Put  MC dt = 39.280919
Call abs error = 0.898564
Put  abs error = 0.016165
Call 95% CI    = (51.453027, 54.007841)
Put  95% CI    = (39.086708, 39.475130)
